In [ ]:
# =====================================================
# MODEL TRENDING NOTEBOOK
# Tracks model performance and interpretability over time
# =====================================================

# Install dependencies if needed (for Colab)
!pip install pandas numpy matplotlib scikit-learn -q

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

# Clone the repo
REPO_URL = "https://github.com/sad19016/myscrapers-sad19016.git"
REPO_NAME = "myscrapers-sad19016"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    # Pull latest changes if already cloned
    !git -C {REPO_NAME} pull

RESULTS_DIR = f"{REPO_NAME}/results"
print(f"Results directory: {RESULTS_DIR}")
print(f"Prediction files found: {len(glob.glob(f'{RESULTS_DIR}/*-preds.csv'))}")
print(f"Importance files found: {len(glob.glob(f'{RESULTS_DIR}/*-permutation_importance.csv'))}")
print(f"PDP files found: {len(glob.glob(f'{RESULTS_DIR}/*-pdp_top3.png'))}")

In [ ]:
def calculate_metrics(df):
    """Calculate MAE, MAPE, RMSE, and Bias from a predictions dataframe."""
    # Drop rows where actual_price is missing
    df = df.dropna(subset=["actual_price", "pred_price"])
    if len(df) == 0:
        return None
    
    actual = df["actual_price"]
    predicted = df["pred_price"]
    
    mae = np.mean(np.abs(actual - predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    bias = np.mean(predicted - actual)  # positive = overestimating, negative = underestimating
    
    return {"mae": mae, "mape": mape, "rmse": rmse, "bias": bias}

# Load all prediction files and calculate metrics for each run
metrics_rows = []
pred_files = sorted(glob.glob(f"{RESULTS_DIR}/*-preds.csv"))

for filepath in pred_files:
    filename = os.path.basename(filepath)
    run_id = filename.replace("-preds.csv", "")
    
    try:
        df = pd.read_csv(filepath)
        metrics = calculate_metrics(df)
        if metrics:
            metrics["run_id"] = run_id
            metrics["timestamp"] = pd.to_datetime(run_id, format="%Y%m%d%H")
            metrics["n_predictions"] = len(df)
            metrics_rows.append(metrics)
    except Exception as e:
        print(f"Skipping {filename}: {e}")

metrics_df = pd.DataFrame(metrics_rows).sort_values("timestamp")
print(f"Loaded {len(metrics_df)} runs")
display(metrics_df)

In [ ]:
# Load all permutation importance files
importance_runs = {}
importance_files = sorted(glob.glob(f"{RESULTS_DIR}/*-permutation_importance.csv"))

for filepath in importance_files:
    filename = os.path.basename(filepath)
    run_id = filename.replace("-permutation_importance.csv", "")
    try:
        df = pd.read_csv(filepath)
        importance_runs[run_id] = df
    except Exception as e:
        print(f"Skipping {filename}: {e}")

print(f"Loaded {len(importance_runs)} importance files")

In [ ]:
# =====================================================
# SECTION 3: MODEL ACCURACY DASHBOARD
# =====================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Model Accuracy Over Time", fontsize=16, fontweight="bold")

metrics_to_plot = [
    ("mae", "MAE (Mean Absolute Error)", "blue", "Lower is better"),
    ("mape", "MAPE (Mean Absolute % Error)", "orange", "Lower is better"),
    ("rmse", "RMSE (Root Mean Squared Error)", "red", "Lower is better"),
    ("bias", "Bias (+ = Overestimate, - = Underestimate)", "green", "Closer to 0 is better"),
]

for ax, (metric, title, color, subtitle) in zip(axes.flatten(), metrics_to_plot):
    ax.plot(metrics_df["timestamp"], metrics_df[metric], marker="o", color=color, linewidth=2)
    ax.set_title(f"{title}\n({subtitle})", fontsize=11)
    ax.set_xlabel("Run Date")
    ax.set_ylabel(metric.upper())
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("accuracy_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Accuracy dashboard saved!")

In [ ]:
# Summary statistics table
print("=" * 50)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 50)
summary = metrics_df[["run_id", "mae", "mape", "rmse", "bias", "n_predictions"]].copy()
summary["mae"] = summary["mae"].round(2)
summary["mape"] = summary["mape"].round(2)
summary["rmse"] = summary["rmse"].round(2)
summary["bias"] = summary["bias"].round(2)
display(summary)
print(f"\nBest MAE: ${metrics_df['mae'].min():,.2f} on run {metrics_df.loc[metrics_df['mae'].idxmin(), 'run_id']}")
print(f"Latest MAE: ${metrics_df['mae'].iloc[-1]:,.2f}")

In [ ]:
# =====================================================
# SECTION 4: FEATURE IMPORTANCE TRENDING
# =====================================================

# Plot top 15 features for the latest run
latest_run_id = sorted(importance_runs.keys())[-1]
latest_importance = importance_runs[latest_run_id]

# Get top 15 features by importance
top15 = latest_importance.nlargest(15, "importance_mean")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top15["feature"], top15["importance_mean"], 
               xerr=top15["importance_std"], 
               color="steelblue", alpha=0.8, capsize=3)
ax.set_title(f"Top 15 Features by Permutation Importance\n(Latest Run: {latest_run_id})", 
             fontsize=13, fontweight="bold")
ax.set_xlabel("Importance (drop in negative MAE when feature is shuffled)")
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("feature_importance_latest.png", dpi=150, bbox_inches="tight")
plt.show()
print("Feature importance chart saved!")

In [ ]:
# Track how top 5 features change across runs
if len(importance_runs) > 1:
    # Get top 5 features from latest run
    top5_features = latest_importance.nlargest(5, "importance_mean")["feature"].tolist()
    
    # Build a DataFrame tracking those features across all runs
    tracking_rows = []
    for run_id, imp_df in sorted(importance_runs.items()):
        for feat in top5_features:
            match = imp_df[imp_df["feature"] == feat]
            if not match.empty:
                tracking_rows.append({
                    "run_id": run_id,
                    "timestamp": pd.to_datetime(run_id, format="%Y%m%d%H"),
                    "feature": feat,
                    "importance_mean": match["importance_mean"].values[0]
                })
    
    tracking_df = pd.DataFrame(tracking_rows)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    for feat in top5_features:
        feat_df = tracking_df[tracking_df["feature"] == feat]
        ax.plot(feat_df["timestamp"], feat_df["importance_mean"], 
                marker="o", linewidth=2, label=feat)
    
    ax.set_title("Top 5 Feature Importance Over Time", fontsize=13, fontweight="bold")
    ax.set_xlabel("Run Date")
    ax.set_ylabel("Importance Mean")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("feature_importance_trending.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Only one run available — feature trending will show once more runs complete!")
    print("Run again tomorrow after 10:45 AM to see trending.")

In [ ]:
# =====================================================
# SECTION 5: PARTIAL DEPENDENCE PLOTS (PDPs)
# =====================================================

# Display the latest PDP image
pdp_files = sorted(glob.glob(f"{RESULTS_DIR}/*-pdp_top3.png"))

if pdp_files:
    latest_pdp = pdp_files[-1]
    latest_pdp_run = os.path.basename(latest_pdp).replace("-pdp_top3.png", "")
    
    print(f"Displaying PDP for latest run: {latest_pdp_run}")
    img = mpimg.imread(latest_pdp)
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Partial Dependence Plots — Top 3 Numeric Features\n(Run: {latest_pdp_run})", 
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("pdp_latest.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No PDP files found yet!")

In [ ]:
# If multiple runs exist, show how PDPs changed over time
if len(pdp_files) > 1:
    print(f"Showing PDPs across {len(pdp_files)} runs:")
    fig, axes = plt.subplots(len(pdp_files), 1, 
                              figsize=(15, 5 * len(pdp_files)))
    
    # Handle case of only 2 runs
    if len(pdp_files) == 2:
        axes = [axes[0], axes[1]]
    
    for ax, pdp_file in zip(axes, pdp_files):
        run_id = os.path.basename(pdp_file).replace("-pdp_top3.png", "")
        img = mpimg.imread(pdp_file)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"PDPs for Run: {run_id}", fontsize=11, fontweight="bold")
    
    plt.suptitle("PDP Evolution Over Time", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("pdp_trending.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Only one run available — PDP trending will show once more runs complete!")
    print("Run again tomorrow after 10:45 AM to see how PDPs change over time.")